# DynaGuide Pipeline 
This pipeline was developed and tested for the mimicgen Square_d0 datasat as well as datasets based on the NutAssemblySquare task class.

Make sure you have your conda enviroment activated for the following commands to run.</br>
(When using VSCode, select the respective conda env. as a kernel)
</br></br>
This notebook expects that you have already successfully obtained
- a suitable dataset including train & val splits
- a Diffusion Policy trained on this dataset, respectively.

To run DynaGuide, you need

1) a trained base policy (see previous notebook)
1) guidance conditions (h5 files)
1) a trained dynamics model and
1) the experiment config file 


**We will create all  nescessary files in the following steps.**</br></br>
Once you have these components, you can run run_dynaguide.py.

You can find all nescessary files in the LRZ Sync&share mentioned in the [README](README.md)

# 1. Prerequisites

## 1.1 (If not done yet) Setup DynaGuide environment

### 1.1.1 Create conda environment
Run the following commands in your terminal to create and activate the DynaGuide conda environment:
```bash
conda env create -f dynaguide_env.yml
conda activate dynaguide_env
```

### 1.1.2 Install DynaGuide
After activating the environment, install DynaGuide using pip:
```bash
./setup_environment.sh
```


---
# 2. Dataset Preparation

Take the dataset obtained in  [Dataset Preparation](Dataset_Preparation.ipynb) which was also used to train your diffusion policy. We expect your dataset to already have a train/validation split.

## 2.1 Dataset for Dynamics Model

To train the dynmaics model for DynaGuide you have to prepare your dataset accordingly. 

The following steps have to be carried out: 

First you need to split the dataset into a seperate training and validation dataset file. After this it has to be convert it into the same labeled hdf5 format as the Calvin dataset. This mainly means to bundle all proprioception attributes into a single key named `proprio` to work with DynaGuides native training environment. The folloiwng script combines all nescessary steps for you:

* **Train/Val Splitting:** Separate the input HDF5 file into `square_train.hdf5` and `square_valid.hdf5` using specified filter masks.
* **Camera Filtering:** Keeps all camera image keys by default, or optionally filters for specific camera views.
* **Proprioception Processing:** Copies existing proprioception data or automatically constructs it from low-dimensional states (`robot0_eef_pos`, `robot0_eef_quat`, `robot0_gripper_qpos`). It mainly bundles all proprioception attributes into a single key named `proprio` which is necessary to match DynaGuides expected h5 format.


Use the follwing command to perform all the above steps:

In [ ]:
!python helper_scripts/create_dynamics_dataset.py --input dataset/square_d0/square_d0_visual.hdf5 --out_dir dataset/square_splits/

### 2.1.1 (Opt) Use the following command to change the split:
If your dataset currently has no train / val split, use this script to create one (keep in mind that you should us the same split used to train the DP)
```bash
python robomimic/robomimic/scripts/split_train_val.py \
    --dataset dataset/square_d0/square_d0_visual.hdf5 \
    --ratio 0.1
```

## 2.2 Create Guidance Dataset

You need successfull and unsuccessfull representations of your task to create meaningfull guidance conditions for DynaGuide to steer your diffusion policy.

We provide a set of 50 guidance conditions for each direction. See the LRZ sync&share described in the [README](README.md).</br>
Inside the folder '3. Guidance conditions' you can find guidance conditions with varying sizes and for all three insertion directions.

### 2.2.1 Create successfull samples
First add the DynaGuide specific 9D proprio vector to this dataset:

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits/square_valid.hdf5

Now extract the step from each sample at which successful task completion was registered, in order to create a dataset consisting only of successful samples. For guidance, the external camera view is recommended, omit the gripper camera.

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/square_splits/square_valid.hdf5 \
    --output dataset/square_splits/square_pos_examples.hdf5 \
    --num_demos 20 \
    --camera agentview_image \
    --snapshot_mode success \
    --behavior success

If you select `success` the script creates four files: One file containing all guidance condtions extracted from all inserted samples. The other three are already split for the direection of nut insertion: Left, Back and Right.

### 2.2.2 Create unsuccessfull samples

To obtain ''bad'' data, we use a bad perfroming diffusion policy. For example, you can use a early checkpoint of your diffusion policy training. If you have none, we also provide one in our LRZ Sync&Share (see prevoius notebooks)

First, you have to rollout the policy and enrich the rollouts with the dynaguide specific 'proprio' key.

**Rollout ''bad'' policy and create a dataset**

This usually has to be only done one time. Once you have a .hdf5 dataset of bad rollouts, you can adjust the resulting dataset using the scripts in the next setp.</br>
This script saves the rollouts in the given --dataset_path.</br>
Choose a model that is able to grip the nut, but then fails at inserting it! Do _not_ guide on failed nut grippings!</br>

In [ ]:
!python robomimic/robomimic/scripts/run_trained_agent.py \
    --agent PATH_TO_MODEL.pth \
    --n_rollouts 20 \
    --horizon 200 \
    --dataset_path dataset/square_splits/square_neg_examples_dataset.hdf5 \
    --dataset_obs \
    --env NutAssemblySquare

You can check the rollouts using the following command:

In [ ]:
!python robomimic/robomimic/scripts/playback_dataset.py \
    --dataset dataset/square_splits/square_neg_examples_dataset.hdf5 \
    --video_path expert_demo.mp4 \
    --n 5

Now add the DynaGuide specific 9D proprio vector to this dataset. In case there were small floating point precision overshoots during rollout, those also get clipped automatically:

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits/square_neg_examples_dataset.hdf5

: 

Finally extract the specific frame out of each rollout, where the gripper releases the nut to find ''bad'' insertions. As a reference pass valid samples, so that the script can determine the difference between a good and bad executin on its own.</br>
Make sure to set `--behavior` to **failure**.

In [ ]:
!python helper_scripts/create_guidance_dataset.py \
    --input dataset/square_d0_bad_performance.hdf5 \
    --calibration_input dataset/square_splits/square_valid.hdf5 \
    --output dataset/square_splits/square_neg_examples_gripper_20.hdf5 \
    --camera agentview_image \
    --num_demos 20 \
    --snapshot_mode peg_center_fail \
    --behavior failure

Always make sure to visually inspect the resulting snap shots. The deterministic script may not always find sutiable "bad" insertions. Choose them by hand and use the methods provided in **2.2.4 Trim Guidance Datasets** to adjust them to your needs!

### 2.2.3 Trim Guidance Datasets
After having created the desired & undesired guidance examples we have to further trim those datasets to our needs.</br>
Our own evaluations have shown that using a dataset size of ~20 samples per desired state achieves the best results. Therefore, you might have to shrink the size of your guidance sets.</br>
For guidance on inserting the nut we used a total of 60 positive samples (20 per dierection).

**Positive guidance**</br>
First view the created dataset to identify samples you want to remove

In [ ]:
!python helper_scripts/show_guidance_images.py \
    --dataset dataset/square_splits/square_pos_examples.hdf5 \
    --output evaluation/pos_images_extracted

Identify all samples you want to delete. The resulting dataset should contain:
- only ~20 samples of successfull completion, right in the moment, where the nut is inserted

Now remove all idientified samples. Use the numbers for the i-th sample you want to remove. Remember, that the samples start counting at 0! So the 5th sample actually corresponds to the number 4. 
Here, the samples with the number 2, 5, 10-30, 40, 51 are removed from a recuded set of samples remains. 

In [ ]:
!python helper_scripts/filter_guidance_dataset.py \
    --input dataset/square_splits/square_pos_examples.hdf5 \
    --exclude 2 5 10-30 40 51

NOTE: After creating a guidance set, the resulting dataset it can only be analyzed using `python robomimic/robomimic/scripts/get_dataset_info.py --dataset DATASET` anymore! `show_hdf5.py` or `analyze_dataset.py` fail as the guidance datasets have a far reduced strcture.

If you now want to merge several datasets together, eg. left, right and back insertion guidance examples, use
```bash
python concat_datasets.py \
    --train_dataset path/to/dataset1.hdf5 \
    --val_dataset path/to/dataset2.hdf5 \
    --additional_dataset path/to/dataset3.hdf5 \
    --output path/to/concatenated_output.hdf5
```

**Negative guidance**

The negative guidance samples should capture clear, task-relevant failure states at the exact moment the mistake becomes irrecoverable. Eg:
- Jammed: The nut is resting diagonally on top of the square peg, or caught on the rim, completely un-inserted. The gripper may still be touching it or just beginning to release.
- Missed: The nut has slipped completely past the peg and is sitting flat on the white table surface next to the base, with the gripper hovering nearby empty-handed.

Important: The robot arm's pixel footprint must be nearly identical to your positive samples. The only visual difference should be what the nut is doing. Else, the guidance will steer the DP to avoide sections of the table completely!

Here are a few example pictures of positive guidance states, that we have used in our project:

<img src="pictures/guidance_example.png" height="267">

After running the filter, extract the resulting dataset using `show_guidance_images` once more to verify succesfull filtering. 

### 2.2.4 Create reset poses
If the run_dynaguide.py script requires this file to launch your Robosuite environment, you can generate a compatible initial state layout by extracting the starting states from your validation dataset.
Pay atention: Use the same dataset than you later use for run_dynaguide!

In [ ]:
!python helper_scripts/create_reset_poses.py \
    --input dataset/square_splits/square_valid.hdf5 \
    --output dataset/square_splits/square_initial_states.json

#### 2.2.2.5 Create dynaguide_env_config.yml
The final config should look like this. Copy the format and insert the paths to your resulting gudiance & reset position files to later successfully run DynaGuide.

```json
{ 
    "env_setup": {},
    "use_neg": true, 
    "pos_examples": "PATH_TO_POS_SAMPLES.hdf5", 
    "neg_examples": "PATH_TO_NEG_SAMPLES.hdf5",
    "reset_poses": "PATH_TO_RESET_POSITIONS.json"
}  
```

---
# 3. Train Base Policy

See the notebook [DP_Training](DP_Training.ipynb) to obtain a suitable base policy.

This base policy will then be fine tuned and improved using DynaGuide steering in the next setps.

---
# 4. Train Dynamics Model

## 4.1.1 Train the base policy with the square_d0_visual dataset

Use the following command to train the dynamics model:

```bash
python dyna_scripts/train_dynaguide.py \
    --exp_dir result_16 \
    --train_hdf5 dataset/square_splits/square_train.hdf5 \
    --test_hdf5 dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 \
    --proprio_key proprio \
    --proprio_dim 9 \
    --num_epochs 1000 \
    --action_chunk_length 16 \
    --batch_size 32 \
    --eval_every 1 \
    --save_every 5 \
    --steps_per_epoch 200
```


In [ ]:
!python dyna_scripts/train_dynaguide.py \
    --exp_dir result_16 \
    --train_hdf5 dataset/square_splits/square_train.hdf5 \
    --test_hdf5 dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 \
    --proprio_key proprio \
    --proprio_dim 9 \
    --num_epochs 1000 \
    --action_chunk_length 16 \
    --batch_size 32 \
    --eval_every 1 \
    --save_every 5 \
    --steps_per_epoch 200

and test it using the validation dataset

```bash
python dyna_scripts/test_dynaguide_embedding.py \
    --exp_dir result_16 \
    --dataset dataset/square_splits/square_valid.hdf5 \
    --checkpoint_dir result_16/checkpoints/ \
    --camera agentview_image \
    --action_dim 7 \
    --proprio_dim 9
```

In [ ]:
!python dyna_scripts/test_dynaguide_embedding.py \
    --exp_dir result_16 \
    --dataset dataset/square_splits/square_valid.hdf5 \
    --checkpoint_dir result_16/checkpoints/ \
    --camera agentview_image \
    --action_dim 7 \
    --proprio_dim 9

### 4.1.2 Add actions created from bad base policy rollouts to learn bad actions

(Before you can do this you need to do a couple of rollouts with models from early stages of the DP training - see )

```bash
python robomimic/robomimic/scripts/run_trained_agent.py \
    --agent PATH_TO_MODEL.pth \
    --dataset_path /path/to/bad_examples.hdf5 \
    --horizon 200 \
    --n_rollouts 10 \
```

1. Convert the Rollouts to into square_d0_visual format

```bash
python helper_scripts/convert_rollout_to_dataset.py \
    --input path/to/rollout \
    --output dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5
```

In [ ]:
!python helper_scripts/convert_rollout_to_dataset.py \
    --input path/to/rollout \
    --output dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5

2. Merge the datasets

`merge_datasets.py` preserves the original train/val split: it copies `mask/valid` from the base dataset unchanged, and appends only the new rollout episodes to `mask/train`. `--shuffle` only randomizes the order in which the rollout episodes themselves are appended — it does not touch the base dataset's episodes or its existing split.

```bash
python helper_scripts/merge_datasets.py \
  --base dataset/square_d0/square_d0_visual.hdf5 \
  --rollout \
    dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5 \
    dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5 \
    dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5 \
    dataset/square_d0_bad_actions/rollouts_converted_ep_.hdf5 \
    ...
  --output dataset/square_d0/square_d0_with_rollouts.hdf5 \
  --shuffle
```

Do **not** re-run `split_train_val.py` on the merged output

3. Create the Dynamic Model train and val dataset again.

Since the merged dataset already carries the correct `mask/train` and `mask/valid` (original split preserved), this just re-materializes the two split files from that mask — no new randomization happens here.

```bash
python helper_scripts/create_dynamics_dataset.py --input dataset/square_d0/square_d0_with_rollouts.hdf5 --out_dir dataset/square_splits_augmented/
```

In [ ]:
!python helper_scripts/create_dynamics_dataset.py --input dataset/square_d0/square_d0_with_rollouts.hdf5 --out_dir dataset/square_splits_augmented/

4. Add Proprio keys to the new datasets


```bash
python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits_augmented/square_train.hdf5
python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits_augmented/square_valid.hdf5
```

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits_augmented/square_train.hdf5
!python helper_scripts/add_proprio_to_dataset.py --input dataset/square_splits_augmented/square_valid.hdf5

### 4.1.3 Train the Dynamics Model again


```bash
python dyna_scripts/train_dynaguide.py \
    --exp_dir result_32_augmented \
    --train_hdf5 dataset/square_splits_augmented/square_train.hdf5 \
    --test_hdf5 dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 \
    --proprio_key proprio \
    --proprio_dim 9 \
    --num_epochs 1000 \
    --action_chunk_length 16 \
    --batch_size 16 \
    --eval_every 1 \
    --save_every 5 \
    --steps_per_epoch 200
```

In [ ]:
!python dyna_scripts/train_dynaguide.py \
    --exp_dir result_16_augmented \
    --train_hdf5 dataset/square_splits_augmented/square_train.hdf5 \
    --test_hdf5 dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 \
    --proprio_key proprio \
    --proprio_dim 9 \
    --num_epochs 1000 \
    --action_chunk_length 16 \
    --batch_size 16 \
    --eval_every 1 \
    --save_every 5 \
    --steps_per_epoch 200

---
### 4.1.4 Helpfull — Resume Training or Extend with OOD Data (→ Section 7)

After running the OOD experiments in **Section 7**, you have rollout datasets collected with the `NutAssemblyGuidanceGold` and `NutAssemblyGuidanceRed` environments. These can be fed back into the dynamics model to broaden its visual training distribution. The two workflows below are independent — use either or both.

---

#### A. Resume an interrupted run (same TensorBoard log, same exp directory)

`--resume` auto-detects the highest-epoch checkpoint in `exp_dir/checkpoints/` and continues from there. TensorBoard epoch numbers keep counting up, so loss curves are uninterrupted:

In [ ]:
!python dyna_scripts/train_dynaguide.py \
    --exp_dir result_32_augmented \
    --resume \
    --train_hdf5 dataset/square_splits_augmented/square_train.hdf5 \
    --test_hdf5  dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 --proprio_key proprio --proprio_dim 9 \
    --num_epochs 200 \
    --action_chunk_length 16 --batch_size 16 \
    --eval_every 1 --save_every 5 --steps_per_epoch 200

To resume from a specific checkpoint instead of the latest, replace `--resume` with:
```bash
--resume_checkpoint result_16/checkpoints/model_epoch_.pth
```

> `args.json` is never overwritten on resume. A separate `args_resumed_epochN.json` is written to `exp_dir` so there is a full record of which datasets were used for each training phase.

---

#### B. Continue training with red-square rollouts from Section 7

The `NutAssemblyGuidanceGold` / `NutAssemblyGuidanceRed` rollouts collected in **Section 7.5.1 B** give the dynamics model clean single-colour pickup trajectories it has never seen. First make sure proprio has been added to them (usually already done in 7.5.1 B):

In [ ]:
!python helper_scripts/add_proprio_to_dataset.py --input dataset/dual_square/gold_pickup_rollouts.hdf5
!python helper_scripts/add_proprio_to_dataset.py --input dataset/dual_square/red_pickup_rollouts.hdf5

Pass all files to `--train_hdf5` — the script concatenates them automatically via `ConcatDataset`. Use a new `--exp_dir` and load the best existing checkpoint with `--resume_checkpoint` to fine-tune rather than train from scratch:

In [ ]:
!python dyna_scripts/train_dynaguide.py \
    --exp_dir result_16_with_ood \
    --resume_checkpoint result_16/checkpoints/model_epoch_200.pth \
    --train_hdf5 \
        dataset/square_splits_augmented/square_train.hdf5 \
        dataset/dual_square/gold_pickup_rollouts.hdf5 \
        dataset/dual_square/red_pickup_rollouts.hdf5 \
    --test_hdf5  dataset/square_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 --proprio_key proprio --proprio_dim 9 \
    --num_epochs 200 \
    --action_chunk_length 16 --batch_size 16 \
    --eval_every 1 --save_every 5 --steps_per_epoch 200

At startup the script prints the combined dataset size:
```
[Dataset] Concatenated 3 training files → 18450 samples total:
  dataset/square_splits_augmented/square_train.hdf5
  dataset/dual_square/gold_pickup_rollouts.hdf5
  dataset/dual_square/red_pickup_rollouts.hdf5
```

Using a fresh `exp_dir` keeps the extended run's TensorBoard logs separate from the original, making it easy to compare the two training curves side by side.


---
### 4.1.5 Fine-tune with OOD Rollouts (multi-file train & validation)

`--train_hdf5` and `--test_hdf5` both accept multiple HDF5 files. The script concatenates them via `ConcatDataset` and samples uniformly, so the OOD data appears proportionally to its size.

**Example setup:** 1050 augmented square samples + 150 red-square rollouts → split 90/10 → ~135 for training, ~15 for validation.

**Step 1 — Split the rollout file (90 % train / 10 % val):**

In [ ]:
!python robomimic/robomimic/scripts/split_train_val.py \
    --dataset dataset/dual_square/dual_red_pickup_rollouts_dynamics_training.hdf5 \
    --ratio 0.1

**Step 2 — Create separate train/val HDF5 files from the split:**

In [ ]:
!python helper_scripts/create_dynamics_dataset.py \
    --input dataset/dual_square/dual_red_pickup_rollouts_dynamics_training.hdf5 \
    --out_dir dataset/dual_square/rollout_splits/

This produces `rollout_splits/square_train.hdf5` (~135 samples) and `rollout_splits/square_valid.hdf5` (~15 samples).

**Step 3 — Train with OOD data in both train and val:**

In [ ]:
!PYTHONPATH=. python dyna_scripts/train_dynaguide.py \
    --exp_dir result_16_augmented_200_with_ood \
    --resume_checkpoint result_16_augmented_200/checkpoints/model_epoch_999.pth \
    --train_hdf5 \
        dataset/square_splits_augmented/square_train.hdf5 \
        dataset/dual_square/rollout_splits/square_train.hdf5 \
    --test_hdf5 \
        dataset/square_splits/square_valid.hdf5 \
        dataset/dual_square/rollout_splits/square_valid.hdf5 \
    --cameras agentview_image \
    --action_dim 7 --proprio_key proprio --proprio_dim 9 \
    --num_epochs 200 \
    --action_chunk_length 16 --batch_size 16 \
    --eval_every 1 --save_every 5 --steps_per_epoch 200

At startup the script prints the combined sizes, e.g.:
```
[Dataset] Concatenated 2 training files → ~1185 samples total:
  dataset/square_splits_augmented/square_train.hdf5
  dataset/dual_square/rollout_splits/square_train.hdf5
[Dataset] Concatenated 2 validation files → ... samples total:
  dataset/square_splits/square_valid.hdf5
  dataset/dual_square/rollout_splits/square_valid.hdf5
```


---
# 5. Running DynaGuide

To run DynaGuide, you need 

1) Trained base policy 
1) Trained dynamics model 
1) Guidance conditions (h5 files). 
1) The experiment config file 

Once you have these components, you can run run_dynaguide.py using the following code:

```bash
python dyna_scripts/run_dynaguide.py \
    --output_folder evaluation_/ \
    --horizon 200 \
    --n_rollouts 50 \
    --dp_model trained_models/model_ep_.pth \
    --env NutAssemblySquare \
    --video_skip 2 \
    --exp_setup_config dynaguide_env_config.json \
    --dynamics_model result_16/checkpoints/model_epoch_.pth \
    --camera_names agentview_image \
    --seed 0
```
- Use `--render_denoising` if you want to see the denoising steps on the top of the video. 
- When omitting `--n_rollouts` the script will rollout on all samples included in `reset_poses` in the env_config.json.
- Using `--seed 0` leads to deterministic, not random evaluation. Usefull for true comparisons.

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --output_folder evaluation_/ \
    --horizon 200 \
    --n_rollouts 50 \
    --dp_model trained_models/model_epoch_800_wo_proprio.pth \
    --env NutAssemblySquare \
    --video_skip 2 \
    --exp_setup_config dynaguide_env_config.json \
    --dynamics_model result_16/checkpoints/model_epoch_115.pth \
    --camera_names agentview_image \
    --seed 0


---
# 6. Evalutaion

## 6.1 Evaluating the Diffusion Policy

To compare the diffusion policy with the exakt same samples as the DynaGuide rollout used, run:
```bash
python dyna_scripts/run_trained_agent_in_order.py \
  --agent trained_models/DP_MODEL.pth \
  --horizon 200 \
  --n_rollouts 50 \
  --video_path ./evaluation/trained_agent_in_order/eval_dp.mp4 \
  --dataset_path evaluation/trained_agent_in_order/rollouts.hdf5 \
  --eval_dataset dataset/square_splits/square_valid.hdf5 \
  --env NutAssemblySquare \
  --seed 0
```
`--eval_dataset` is beeing used, to pass the initial positions of the validation samples. They are then evaluated in the order they are listed in the respective validation dataset.

Or use this command, if you pass the initials_states.json instead of the validation dataset.
```bash
python dyna_scripts/run_trained_agent_in_order.py \
  --agent trained_models/DP_MODEL.pth \
  --horizon 200 \
  --n_rollouts 50 \
  --video_path ./evaluation/trained_agent_in_order/eval_dp.mp4 \
  --dataset_path evaluation/trained_agent_in_order/rollouts.hdf5 \
  --initial_states_json square_initial_states.json \
  --env NutAssemblySquare \
  --seed 0
```

## 6.2 Evaluating DynaGuide

Use the following additional parameters to save the DynaGuide rollouts in a HDF5 file to later anaylze them more intensively.
```bash
python dyna_scripts/run_dynaguide.py \
    --output_folder evaluation_2406/eval_robomimic_back_50/ \
    --horizon 200 \
    --n_rollouts 50 \
    --dp_model trained_models/model_epoch_800_wo_proprio.pth \
    --env NutAssemblySquare \
    --video_skip 2 \
    --exp_setup_config dynaguide_env_config.json \
    --dynamics_model trained_models/result100.pth \
    --camera_names agentview_image \
    --seed 0 \
    --render_denoising \
    --dataset_obs \
    --save_images 
```

In [ ]:
!python dyna_scripts/run_dynaguide.py \
    --output_folder evaluation_2406/eval_robomimic_back_50/ \
    --horizon 200 \
    --n_rollouts 50 \
    --dp_model trained_models/model_epoch_800_wo_proprio.pth \
    --env NutAssemblySquare \
    --video_skip 2 \
    --exp_setup_config dynaguide_env_config.json \
    --dynamics_model trained_models/result100.pth \
    --camera_names agentview_image \
    --seed 0 \
    --render_denoising \
    --dataset_obs \
    --save_images 

You can now plot the DynamicsModel latent space predictions using `pca` or `tsne` (optional: --perplexity 30). Makre sure you used `--dataset_obs` to run DynaGuide! Else the latent space encodings wont be saved in the rollout.

```bash
python helper_scripts/plot_rollout_dynamics.py \
  --guided_rollout_path PATH_TO_SAVED_DYNAGUIDE_ROLLOUT.hdf5 \
  --use_saved_embeddings \
  --output_dir WHERE_TO_SAVE_THE_PICTURES \
  --method pca \
  --chunk_length 16
```

---
# Next steps


After having successfully deployed the diffusion policy while simulatneosuly using DynaGuide to steer the policy towards desired end states, you can now conuduct several different expereiments to push DynaGuide's steering to the test.

For this, see [Experiments](Experiments.ipynb).